In [10]:
# WEB SERVER ACCESS LOG ANALYSIS
# Student-specific parameters

import csv
import json
import shutil
import re
from pathlib import Path
from collections import Counter

# Student information
SEED_DIGIT = 9
ID_SUM = 21

# Generated requirements
RECORD_LIMIT = 1000 + ID_SUM
TARGET_STATUS_CLASS = (SEED_DIGIT % 4) + 2

# File locations
INPUT_FILE = Path("weblog.csv")
OUTPUT_FOLDER = Path("output")

# Create output directory
OUTPUT_FOLDER.mkdir(exist_ok=True)

print("WEB SERVER ACCESS LOG ANALYSIS")
print("--------------------------------")
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Records to process:", RECORD_LIMIT)
print("Target Status Class:", str(TARGET_STATUS_CLASS) + "xx")
print("Output folder created successfully!")

WEB SERVER ACCESS LOG ANALYSIS
--------------------------------
Seed Digit: 9
ID Sum: 21
Records to process: 1021
Target Status Class: 3xx
Output folder created successfully!


In [11]:
if not INPUT_FILE.exists():
    print("ERROR: Dataset file not found!")
    print("Please place web_log_dataset.csv")
    print("inside the Exercise 1 folder.")

else:
    with open(INPUT_FILE, "r", newline="", encoding="utf-8-sig") as file:
        reader = csv.DictReader(file)

        print("CSV file found successfully!")
        print("Column names:", reader.fieldnames)

        print("\nFirst 3 records:")

        for index, row in enumerate(reader):
            if index == 3:
                break
            print(row)

CSV file found successfully!
Column names: ['IP', 'Time', 'URL', 'Staus']

First 3 records:
{'IP': '10.128.2.1', 'Time': '[29/Nov/2017:06:58:55', 'URL': 'GET /login.php HTTP/1.1', 'Staus': '200'}
{'IP': '10.128.2.1', 'Time': '[29/Nov/2017:06:59:02', 'URL': 'POST /process.php HTTP/1.1', 'Staus': '302'}
{'IP': '10.128.2.1', 'Time': '[29/Nov/2017:06:59:03', 'URL': 'GET /home.php HTTP/1.1', 'Staus': '200'}


In [12]:

# CELL 3: PROCESS AND FILTER WEB SERVER LOGS

import csv
import re
from collections import Counter
from pathlib import Path

# Student-specific parameters
SEED_DIGIT = 9
ID_SUM = 21

RECORD_LIMIT = 1000 + ID_SUM
TARGET_STATUS_CLASS = (SEED_DIGIT % 4) + 2

# Input and output files
INPUT_FILE = Path("weblog.csv")
OUTPUT_FOLDER = Path("output")
OUTPUT_FOLDER.mkdir(exist_ok=True)

# Initialize counters
status_counts = Counter()
url_counts = Counter()

records_processed = 0
selected_count = 0
invalid_records = 0

# Output CSV file
output_file = OUTPUT_FOLDER / "filtered_logs.csv"

# Check if dataset exists
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Dataset not found: {INPUT_FILE.resolve()}"
    )

# Open original CSV
with open(INPUT_FILE, "r", newline="", encoding="utf-8-sig") as file:

    reader = csv.DictReader(file)
    headers = reader.fieldnames

    if not headers:
        raise ValueError("CSV file has no column headers.")

    # Identify required columns
    def find_column(headers, possible_names):
        for header in headers:
            normalized = re.sub(
                r"[^a-z0-9]", "", header.lower()
            )

            if normalized in possible_names:
                return header

        return None

    ip_col = find_column(headers, ["ip", "ipaddress"])
    time_col = find_column(headers, ["time", "timestamp"])
    url_col = find_column(
        headers, ["url", "requestedurl", "requesturl"]
    )
    status_col = find_column(
        headers,
        ["status", "staus", "httpstatus", "httpstatuscode"]
    )

    if not all([ip_col, time_col, url_col, status_col]):
        raise ValueError(
            f"Required columns missing. Found: {headers}"
        )

    # Create filtered CSV
    with open(
        output_file, "w", newline="", encoding="utf-8"
    ) as output:

        writer = csv.DictWriter(
            output, fieldnames=headers
        )
        writer.writeheader()

        # Read only the first 1021 data records
        for row in reader:

            if records_processed >= RECORD_LIMIT:
                break

            records_processed += 1

            # Extract original fields
            ip = (row.get(ip_col) or "").strip()
            timestamp = (row.get(time_col) or "").strip()
            request = (row.get(url_col) or "").strip()
            status = (row.get(status_col) or "").strip()

            # Validate required fields
            if not all([ip, timestamp, request, status]):
                invalid_records += 1
                continue

            # Extract HTTP status code
            match = re.search(r"\b(\d{3})\b", status)

            if not match:
                invalid_records += 1
                continue

            status_code = int(match.group(1))

            # Count valid HTTP status codes
            status_counts[status_code] += 1

            # Extract requested URL from HTTP request
            parts = request.split()

            if not parts:
                invalid_records += 1
                continue

            if parts[0].upper() in [
                "GET", "POST", "PUT", "DELETE",
                "HEAD", "OPTIONS", "PATCH"
            ]:
                if len(parts) < 2:
                    invalid_records += 1
                    continue

                requested_url = parts[1]
            else:
                requested_url = parts[0]

            # Select only target HTTP status class (3xx)
            if status_code // 100 == TARGET_STATUS_CLASS:

                selected_count += 1

                # Count requested URLs
                url_counts[requested_url] += 1

                # Export original record
                writer.writerow(row)

# Identify five most requested URLs
top_five_urls = url_counts.most_common(5)

# Display processing results
print(" WEB SERVER ACCESS LOG ANALYSIS")

print("Student SEED_DIGIT:", SEED_DIGIT)
print("Student ID_SUM:", ID_SUM)
print("Records to process:", RECORD_LIMIT)

print("\nPROCESSING RESULTS")

print("Records processed:", records_processed)
print("Invalid records:", invalid_records)
print("Valid HTTP records:", sum(status_counts.values()))
print("Target status class:", str(TARGET_STATUS_CLASS) + "xx")
print("Records in target class:", selected_count)

print("\nCOUNT PER STATUS CODE")

for code, count in sorted(status_counts.items()):
    print(f"HTTP {code}: {count}")

print("\nFIVE MOST REQUESTED URLS")

if top_five_urls:
    for url, count in top_five_urls:
        print(f"{url}: {count} requests")
else:
    print("No matching URLs found.")

print("\nFiltered CSV created:", output_file)


 WEB SERVER ACCESS LOG ANALYSIS
Student SEED_DIGIT: 9
Student ID_SUM: 21
Records to process: 1021

PROCESSING RESULTS
Records processed: 1021
Invalid records: 0
Valid HTTP records: 1021
Target status class: 3xx
Records in target class: 165

COUNT PER STATUS CODE
HTTP 200: 843
HTTP 206: 8
HTTP 302: 144
HTTP 304: 21
HTTP 404: 5

FIVE MOST REQUESTED URLS
/: 47 requests
/process.php: 35 requests
/contestproblem.php?name=RUET%20OJ%20Server%20Testing%20Contest: 34 requests
/home.php: 9 requests
/logout.php: 7 requests

Filtered CSV created: output\filtered_logs.csv


In [13]:

# CELL 4: CREATE JSON SUMMARY AND BACKUP

import json
import shutil

# Prepare JSON summary
summary = {
    "student_information": {
        "seed_digit": SEED_DIGIT,
        "id_sum": ID_SUM
    },

    "records_processed": records_processed,

    "target_status_class": str(TARGET_STATUS_CLASS) + "xx",

    "records_in_target_class": selected_count,

    "invalid_records": invalid_records,

    "count_per_status_code": dict(status_counts),

    "five_most_requested_urls": [
        {
            "url": url,
            "request_count": count
        }
        for url, count in top_five_urls
    ]
}

# JSON output file
json_file = OUTPUT_FOLDER / "log_summary.json"

# Write JSON summary
with open(json_file, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

print("JSON SUMMARY")
print("JSON file created:", json_file)

# Create JSON backup
backup_file = OUTPUT_FOLDER / "log_summary_backup.json"

shutil.copy2(json_file, backup_file)

# Verify backup
if backup_file.exists():

    # Compare original and backup contents
    with open(json_file, "rb") as original:
        original_content = original.read()

    with open(backup_file, "rb") as backup:
        backup_content = backup.read()

    if original_content == backup_content:
        print("Backup verification: SUCCESS")
        print("Backup file:", backup_file)
    else:
        print("Backup verification: FAILED")

else:
    print("Backup verification: FAILED")



JSON SUMMARY
JSON file created: output\log_summary.json
Backup verification: SUCCESS
Backup file: output\log_summary_backup.json


In [14]:

# CELL 5: FINAL ASSESSMENT RESULTS

print("\n")
print("       FINAL ASSESSMENT RESULTS")

print("\nSTUDENT-SPECIFIC PARAMETERS")
print("SEED_DIGIT:", SEED_DIGIT)
print("ID_SUM:", ID_SUM)

print("\nPROCESSING INFORMATION")
print("Total records processed:", records_processed)
print("Target status class:", str(TARGET_STATUS_CLASS) + "xx")
print("Records in target class:", selected_count)
print("Invalid records:", invalid_records)

print("\nCOUNT PER STATUS CODE")

for code, count in sorted(status_counts.items()):
    print(f"Status {code}: {count} requests")

print("\nFIVE MOST REQUESTED URLS")

if top_five_urls:
    for number, (url, count) in enumerate(top_five_urls, start=1):
        print(f"{number}. {url} - {count} requests")
else:
    print("No matching URLs found.")

print("\nGENERATED FILES")

print("Filtered CSV:", output_file)
print("JSON Summary:", json_file)
print("JSON Backup:", backup_file)

print("\nFILE VERIFICATION")

print("Filtered CSV exists:", output_file.exists())
print("JSON summary exists:", json_file.exists())
print("JSON backup exists:", backup_file.exists())

if output_file.exists():
    print("Filtered CSV size:", output_file.stat().st_size, "bytes")

if json_file.exists():
    print("JSON summary size:", json_file.stat().st_size, "bytes")

if backup_file.exists():
    print("JSON backup size:", backup_file.stat().st_size, "bytes")

print("       PROGRAM COMPLETED SUCCESSFULLY")




       FINAL ASSESSMENT RESULTS

STUDENT-SPECIFIC PARAMETERS
SEED_DIGIT: 9
ID_SUM: 21

PROCESSING INFORMATION
Total records processed: 1021
Target status class: 3xx
Records in target class: 165
Invalid records: 0

COUNT PER STATUS CODE
Status 200: 843 requests
Status 206: 8 requests
Status 302: 144 requests
Status 304: 21 requests
Status 404: 5 requests

FIVE MOST REQUESTED URLS
1. / - 47 requests
2. /process.php - 35 requests
3. /contestproblem.php?name=RUET%20OJ%20Server%20Testing%20Contest - 34 requests
4. /home.php - 9 requests
5. /logout.php - 7 requests

GENERATED FILES
Filtered CSV: output\filtered_logs.csv
JSON Summary: output\log_summary.json
JSON Backup: output\log_summary_backup.json

FILE VERIFICATION
Filtered CSV exists: True
JSON summary exists: True
JSON backup exists: True
Filtered CSV size: 12083 bytes
JSON summary size: 895 bytes
JSON backup size: 895 bytes
       PROGRAM COMPLETED SUCCESSFULLY
